# EconoNigeria 2.0 – Macroeconomic Intelligence & Forecasting Quickstart

Welcome to the official reproducible research notebook for **EconoNigeria 2.0**.

This notebook demonstrates how to:
1. Query the versioned Open REST API (`/v1/indicators`, `/v1/pulse`, `/v1/signals`)
2. Fetch historical time-series as Pandas DataFrames
3. Run correlation analysis between Currency Devaluation (NGN/USD) and Headline CPI Inflation
4. Inspect automated macroeconomic anomaly signals
5. Evaluate predictive forecast trajectories (Prophet + ARIMA Ensemble)

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

# Base URL for the EconoNigeria public or local API
BASE_URL = "https://econonigeria.org/v1"
# For local development, uncomment below:
# BASE_URL = "http://localhost:8000/v1"

## 1. Monitor the Nigeria Economic Pulse Index
The Pulse Index provides an algorithmic 0–100 composite score tracking macroeconomic resilience.

In [ ]:
response = requests.get(f"{BASE_URL}/pulse")
pulse_data = response.json()

print(f"Composite Pulse Score: {pulse_data['pulse_score']}/100")
print(f"Macro Rating: {pulse_data['pulse_rating']}")
print(f"Executive Summary: {pulse_data['summary']}")

drivers_df = pd.DataFrame(pulse_data['drivers'])
drivers_df

## 2. Ingest Longitudinal Time Series (Inflation & Exchange Rate)
We pull historical observations directly using the `/v1/indicators/{slug}/history` endpoint.

In [ ]:
def get_history_df(indicator_slug: str) -> pd.DataFrame:
    url = f"{BASE_URL}/indicators/{indicator_slug}/history"
    res = requests.get(url)
    data = res.json()
    df = pd.DataFrame(data["history"])
    df["year"] = pd.to_numeric(df["period"], errors="coerce")
    df = df.dropna(subset=["year"]).sort_values("year")
    return df

cpi_df = get_history_df("inflation")
fx_df = get_history_df("exchange-rate")

print(f"CPI Observations: {len(cpi_df)}")
print(f"FX Observations: {len(fx_df)}")

## 3. Visualize Macro Trajectory
Plotting headline inflation rate against historical milestones.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(cpi_df["year"], cpi_df["value"], marker="o", color="#b91c1c", linewidth=2, label="Headline CPI (%)")
plt.axhline(9.0, color="green", linestyle="--", alpha=0.7, label="CBN Upper Target Corridor (9%)")
plt.title("Nigeria Consumer Price Inflation Trajectory", fontsize=14, fontweight="bold")
plt.xlabel("Observation Period")
plt.ylabel("Annual Rate (%)")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Query Automated Economic Signals
Check currently active structural alerts, monetary warnings, and commodity stabilizers.

In [ ]:
signals_res = requests.get(f"{BASE_URL}/signals")
signals = signals_res.json()["signals"]

for s in signals:
    print(f"[{s['severity'].upper()}] {s['title']} ({s['metric_value']} vs {s['benchmark']})")
    print(f"   Policy Implication: {s['implication']}\n")

## 5. Multi-Horizon Forecasts & Accuracy Metrics
Inspect machine learning forecasts (Prophet + ARIMA) and backtesting evaluation (RMSE / MAPE).

In [ ]:
forecast_res = requests.get(f"{BASE_URL}/forecasts/inflation?periods=5&model=ensemble")
fc = forecast_res.json()

print(f"Model Applied: {fc['model_used']}")
print(f"Backtest Evaluation: {fc['evaluation']}")
forecast_df = pd.DataFrame(fc["forecast"])
forecast_df